# 04 질문 라우팅과 RAG 서브에이전트 — 메인이 매뉴얼 담당을 도구로 부르기

02의 기본 RAG는 모든 질문에 매뉴얼을 검색했습니다. 여기서는 역할을 나눕니다.

- **매뉴얼 담당 에이전트(서브에이전트)**: `search_manual` 도구로 현행 매뉴얼을 검색하고, 검색된 조각만 근거로 답과 근거 조각 ID를 돌려줍니다.
- **메인 에이전트**: 인사에는 직접 답하고, 매뉴얼 질문은 `ask_manual_agent` 도구로 담당에게 **위임**합니다.

두 에이전트에 기록용 미들웨어를 붙여 **메인의 위임 기록**과 **담당의 검색 기록(횟수·시간·반환 개수)**을 따로 봅니다.

In [ ]:
# 현재 Python 3.12 커널에 실습 패키지를 설치합니다.
# 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행하세요.
%pip install langchain==1.5.2 langchain-core==1.8.4 langchain-openai==1.8.5 langchain-chroma==1.2.0 chromadb==1.5.9 langchain-text-splitters==1.2.0 langgraph==1.2.24 openai==3.59.0 pypdf==7.1.1 numpy==2.5.5 pandas==3.1.1 scikit-learn==1.9.1 plotly==7.2.0 nbformat==5.11.1 ipykernel==7.4.0

In [ ]:
# ▶ 설정 셀 — 따옴표 안의 값만 바꾸세요. 모델명은 확정값이라 바꾸지 않습니다.
# 키가 든 이 셀은 화면 공유·캡처하거나 AI 도우미에 붙여넣지 않습니다. 파일을 공유할 때는 자리표시 문자열로 되돌립니다.
API_KEY = "여기에_수업용_API_키_입력"
BASE_URL = "여기에_채팅용_BASE_URL_입력"                # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                              # 채팅·도구 호출·에이전트 모델(확정값)
EMBEDDING_BASE_URL = "여기에_임베딩용_BASE_URL_입력"    # 안내받은 임베딩 모델 주소(https://..., 채팅 주소와 다름)
EMBEDDING_MODEL_NAME = "text-embedding-3-small"        # 문서·질문 임베딩 모델(확정값)

In [ ]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
NEEDED = ["API_KEY", "BASE_URL", "EMBEDDING_BASE_URL"]
missing = [name for name in NEEDED if globals()[name].startswith("여기에") or not globals()[name].strip()]
if missing:   # 임베딩 주소가 없을 때 채팅 주소로 대신하지 않는다
    raise ValueError(", ".join(missing) + "에 안내받은 값을 설정 셀의 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1)            # 채팅 주소
json_llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1,
                      model_kwargs={"response_format": {"type": "json_object"}})   # JSON 객체 하나만 답하는 모델
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, api_key=API_KEY, base_url=EMBEDDING_BASE_URL,   # 임베딩 주소
                              check_embedding_ctx_length=False)
print("채팅 모델:", MODEL_NAME, "· 임베딩 모델:", EMBEDDING_MODEL_NAME)

In [ ]:
import hashlib, re
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter

TITLE = "교대 인계 메모 작성·정정 매뉴얼"
MANUALS = [{"file": "data/교대인계메모_매뉴얼_현행본_제3판.pdf", "revision": "현행", "edition": "제3판"},
           {"file": "data/교대인계메모_매뉴얼_구판_제2판.pdf", "revision": "구판", "edition": "제2판"}]
DROP = [r"^교대 인계 메모 작성·정정 매뉴얼 · ", r"^교육용 가상 규정", r"^(현행본|구판) 제\d판 · \d+ / \d+쪽$"]
HEADING, NEW_UNIT = r"^\d+(\.\d+)*\.?\s", r"^(\(\d+\)|판본:)"

def squash(text):                       # 공백·줄바꿈을 지운 문자열(인용 대조용)
    return "".join(str(text).split())

def clean_page_text(raw):               # 머리말·쪽 표시 줄 제거, PDF 줄바꿈으로 끊긴 문장 잇기
    units = []
    for line in (raw or "").splitlines():
        line = line.strip()
        if not line or any(re.search(p, line) for p in DROP):
            continue
        if not units or re.match(HEADING, line) or re.match(NEW_UNIT, line):
            units.append(line)
        else:
            units[-1] += " " + line
    return "\n".join(units)

def read_pages():
    return [{**m, "page": n, "text": clean_page_text(p.extract_text())}
            for m in MANUALS for n, p in enumerate(PdfReader(m["file"]).pages, start=1)]

def split_section(text, size):          # 절 제목에서 새 조각, 길면 (1)·(2) 문단 경계에서 나눔
    chunks, current, has_body = [], [], False
    for unit in text.split("\n"):
        heading = bool(re.match(HEADING, unit))
        if current and has_body and (heading or len("\n".join(current + [unit])) > size):
            chunks.append("\n".join(current)); current, has_body = [], False
        current.append(unit); has_body = has_body or not heading
    return chunks + (["\n".join(current)] if current else [])

def split_fixed(text, size, overlap):   # 정해진 글자 수마다 자름(문장 중간도 자름)
    return CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=overlap).split_text(text)

def make_documents(method="section", size=400, overlap=0):
    docs = []
    for p in read_pages():
        pieces = split_fixed(p["text"], size, overlap) if method == "fixed" else split_section(p["text"], size)
        for n, piece in enumerate(pieces, start=1):
            cid = f"{p['revision']}-p{p['page']}-c{n:02d}"
            docs.append(Document(page_content=piece, id=cid, metadata={
                "chunk_id": cid, "title": TITLE, "revision": p["revision"], "edition": p["edition"],
                "page": p["page"], "file": Path(p["file"]).name}))
    return docs

def collection_name(method, size, overlap):   # 설정·PDF·임베딩 모델이 같으면 같은 이름
    h = hashlib.sha256(EMBEDDING_MODEL_NAME.encode())
    for m in MANUALS:
        h.update(Path(m["file"]).read_bytes())
    return f"manual-{method}-{size}-{overlap}-{h.hexdigest()[:8]}"

def open_index(method="section", size=400, overlap=0):
    """컬렉션을 열고, 아직 없는 조각만 임베딩해 추가한다(같은 설정으로 다시 실행해도 중복 없음)."""
    name = collection_name(method, size, overlap)
    store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                   collection_metadata={"hnsw:space": "cosine"})
    docs = make_documents(method, size, overlap)
    have = set(store.get(ids=[d.id for d in docs])["ids"])
    new = [d for d in docs if d.id not in have]
    if new:
        store.add_documents(new, ids=[d.id for d in new])
    print(f"[색인] {name} · 새로 추가 {len(new)}개 · 기존 재사용 {len(have)}개 · 전체 {store._collection.count()}개")
    return store

def search(store, question, k=3, revision="현행"):
    found = store.similarity_search_with_score(question, k=k, filter={"revision": revision} if revision else None)
    return [{**d.metadata, "text": d.page_content, "distance": round(float(s), 4)} for d, s in found]

def show_hits(hits):
    if not hits:
        print("(검색은 실행됐지만 조건에 맞는 조각이 0개입니다)")
    for rank, h in enumerate(hits, start=1):
        print(f"{rank}위 [{h['chunk_id']}] {h['edition']}({h['revision']}) · PDF {h['page']}쪽 · 거리 {h['distance']}")
        print("   " + h["text"].replace("\n", "\n   "))

In [ ]:
store = open_index("section", 400, 0)

## 1. 기록용 미들웨어

도구가 실행될 때마다 **누가(메인/담당) · 어떤 도구를 · 어떤 입력으로 · 몇 초 동안 · 몇 개를 돌려받았는지** 기록합니다. 질문마다 `LOG`를 비우고 시작합니다.

In [ ]:
import time
from langchain.agents.middleware import AgentMiddleware

LOG = {"calls": [], "hits": [], "sources": []}

class ToolLog(AgentMiddleware):
    def __init__(self, who):
        super().__init__()
        self.who = who
    def wrap_tool_call(self, request, handler):
        t0 = time.perf_counter()
        result = handler(request)
        artifact = getattr(result, "artifact", None)
        LOG["calls"].append({"주체": self.who, "도구": request.tool_call["name"], "입력": request.tool_call["args"],
                             "시간(초)": round(time.perf_counter() - t0, 2),
                             "반환 개수": len(artifact) if isinstance(artifact, list) else "—"})
        return result

MAIN_LOG, MANUAL_LOG = ToolLog("메인"), ToolLog("매뉴얼 담당")

## 2. 매뉴얼 담당 에이전트 — 02의 검색 코드를 도구로 연결

`search_manual`은 01·02의 `search()`를 그대로 부르는 도구입니다. 도구 설명(docstring)은 모델이 도구를 고를 때 읽습니다. 담당 에이전트는 검색한 원문만 근거로 JSON을 돌려줍니다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def search_manual(query: str):
    """교대 인계 메모 매뉴얼(현행 판본)에서 질문과 관련된 원문 조각 3개를 찾는다. query에는 찾을 내용을 한국어 문장으로 넣는다."""
    hits = search(store, query, k=3, revision="현행")
    LOG["hits"].extend(hits)
    text = "\n\n".join(f"[조각ID: {h['chunk_id']}] {h['edition']}({h['revision']}) · PDF {h['page']}쪽\n{h['text']}" for h in hits)
    return text or "검색 결과 없음", hits

MANUAL_AGENT_PROMPT = """당신은 교대 인계 메모 매뉴얼 담당 에이전트입니다.
질문을 받으면 먼저 search_manual 도구로 매뉴얼을 검색합니다(필요하면 표현을 바꿔 한 번 더, 최대 2회).
검색된 원문 조각만 근거로 JSON 객체 하나만 출력합니다.
{"answerable": true 또는 false, "answer": "답변", "citations": [{"chunk_id": "조각ID", "quote": "원문 그대로의 구절"}]}
질문이 묻는 핵심(기간·숫자·담당자 등)이 원문에 없으면 answerable을 false로 하고 answer와 citations를 비웁니다."""

manual_agent = create_agent(model=llm, tools=[search_manual], system_prompt=MANUAL_AGENT_PROMPT, middleware=[MANUAL_LOG])

## 3. 위임 도구와 메인 에이전트

담당 에이전트를 부르는 함수를 `@tool`로 감싸 메인에 등록합니다. 위임 도구는 담당의 JSON을 받아 **인용 조각이 실제 검색 결과에 있고 인용문이 원문에 있을 때만** 답과 출처를 돌려줍니다. 출처는 `LOG`에 남겨 메인의 최종 답과 함께 화면까지 유지합니다.

In [ ]:
import json

NOT_FOUND = "제공된 매뉴얼에서 확인할 수 없습니다."

def read_json(text):
    text = str(text).strip()
    s, e = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[s:e + 1]) if s >= 0 and e > s else {}
    except json.JSONDecodeError:
        return {}

@tool
def ask_manual_agent(question: str) -> str:
    """교대 인계 메모의 작성·정정·확인·보관 등 매뉴얼 내용이 필요한 질문을 매뉴얼 담당 에이전트에게 맡기고, 근거가 확인된 답을 받는다."""
    result = manual_agent.invoke({"messages": [{"role": "user", "content": question}]})
    data = read_json(result["messages"][-1].content)
    by_id = {h["chunk_id"]: h for h in LOG["hits"]}
    cited = [(by_id.get(c.get("chunk_id")), str(c.get("quote", ""))) for c in data.get("citations") or []]
    if not data.get("answerable") or not cited or any(h is None or squash(q) not in squash(h["text"]) for h, q in cited):
        return NOT_FOUND
    for h, q in cited:
        LOG["sources"].append({"chunk_id": h["chunk_id"], "edition": h["edition"], "revision": h["revision"], "page": h["page"], "quote": q})
    marks = ", ".join(f"{h['chunk_id']}({h['edition']} {h['page']}쪽)" for h, _ in cited)
    return f"{data.get('answer', '')}\n[근거: {marks}]"

MAIN_PROMPT = """당신은 교대 인계 업무 도우미의 메인 에이전트입니다.
인사, 감사 인사, 이 챗봇의 사용 방법 질문에는 도구 없이 직접 짧게 답합니다.
인계 메모의 작성·정정·확인·보관 등 매뉴얼 내용이 필요한 질문은 직접 답하지 말고 ask_manual_agent 도구에 질문을 그대로 넘깁니다.
도구가 돌려준 답과 [근거: …] 표시는 바꾸거나 덧붙이지 말고 그대로 전달합니다."""

main_agent = create_agent(model=llm, tools=[ask_manual_agent], system_prompt=MAIN_PROMPT, middleware=[MAIN_LOG])

## 4. 세 질문으로 두 경로 추적하기

질문마다 `LOG`를 비우고 메인을 실행한 뒤, 메인 기록과 담당 기록을 나눠 보여 줍니다.

In [ ]:
import pandas as pd
from IPython.display import display

def run(question):
    LOG.update(calls=[], hits=[], sources=[])
    reply = main_agent.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].content
    main_calls = [c for c in LOG["calls"] if c["주체"] == "메인"]
    searches = [c for c in LOG["calls"] if c["주체"] == "매뉴얼 담당"]
    print(f"질문: {question}")
    print(f"경로: {'매뉴얼 담당에게 위임' if main_calls else '메인이 직접 답변'} · 메인 위임 {len(main_calls)}회 · 담당 검색 {len(searches)}회 "
          f"· 검색 시간 합계 {sum(c['시간(초)'] for c in searches):.2f}초 · 반환 조각 {len(LOG['hits'])}개")
    print("최종 답변:", reply)
    for s in LOG["sources"]:
        print(f"  출처: {s['edition']}({s['revision']}) · PDF {s['page']}쪽 · [{s['chunk_id']}] {s['quote']}")
    if LOG["calls"]:
        display(pd.DataFrame(LOG["calls"]))
    print()

In [ ]:
run("안녕하세요.")

In [ ]:
run("현행 절차에서 인계 메모를 정정할 때 무엇을 기록하고 누가 확인하나요?")

In [ ]:
run("정정 내용을 몇 년 보관하나요?")

## 정리

| 질문 | 메인 기록 | 담당 기록 | 화면 결과 |
|---|---|---|---|
| 안녕하세요. | 위임 0회 | 검색 0회 | 인사 답변, 출처 없음 |
| 현행 정정 절차 | `ask_manual_agent` 1회 | `search_manual` 1회 이상, 반환 3개 | 답 + 근거 조각 ID·판본·쪽 |
| 보관 연수 | `ask_manual_agent` 1회 | `search_manual` 1회 이상, 반환 3개 | 근거 부족 — 검색은 했지만 연수 문장이 없음 |

‘검색 0회’(검색을 안 함)와 ‘검색했지만 근거 부족’은 기록으로 구별됩니다. 실제 검색 횟수는 모델이 정하므로 실행할 때마다 표를 확인합니다.